# DrugMechDB evaluation of NetMedGPT subnetworks

Do the generated subnetworks contain the expert-curated mechanism of action (DrugMechDB) of a drug-disease pair?

- DrugMechDB paths (drug -> protein / biological process / ... -> disease) are mapped to the curated KG:
  drugs by DrugBank ID, diseases MeSH -> MONDO (mondo.obo), proteins UniProt -> NCBI Gene (HGNC), GO, HP, UBERON, Reactome.
  Nodes without a PrimeKG counterpart (pathogen proteins, organisms, chemicals, gene families, cells) are not used.
- Subnetworks: the precomputed consensus subnetworks of generate_subnetwork.py (triplets with frequency >= 0.15).
- Per pair: **mechanism node recall** (curated intermediate nodes found in the subnetwork), **any hit**, **path recovered**
  (all intermediate nodes of a fully mapped path in the subnetwork) and **triplet recall** (curated edges found as subnetwork edges).
- Baselines with the same number of nodes per type as the subnetwork, sampled from (a) the 1-hop KG neighbourhood of the
  drug and the disease and (b) the whole KG; mean over N_RANDOM samples. One-sided Wilcoxon signed-rank test per pair.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "revision_scripts":
    os.chdir("..")
import csv
import glob
import json
import collections
import numpy as np
import pandas as pd
from scipy import stats

DMDB = "data/drugmechdb/indication_paths.json"           # github.com/SuLab/DrugMechDB
HGNC = "data/drugmechdb/hgnc_complete_set.txt"           # UniProt -> NCBI Gene
MONDO = "data/dreamwalk_inputs/mondo.obo"                # MeSH -> MONDO
NODES = "data/KG/nodes_preprocessed.csv"
EDGES = "data/KG/edges_preprocessed.csv"
SUBNET_DIR = "/cosybio_project/netmedgpt_subnetworks_consensus"
N_RANDOM = 20
SEED = 0
OUT_DIR = "data/new_analysis/drugmechdb"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ── Map DrugMechDB identifiers to KG node indices ────────────────────────────
nodes = pd.read_csv(NODES, dtype={"node_id": str})
node_type = dict(zip(nodes["node_index"], nodes["node_type"]))
by_type = {t: dict(zip(g["node_id"], g["node_index"])) for t, g in nodes.groupby("node_type")}

disease_of_mondo = {m: i for nid, i in by_type["disease"].items() for m in nid.split("_")}   # grouped MONDO nodes
mesh_to_mondo, cur = collections.defaultdict(set), None
for line in open(MONDO):
    if line.startswith("id: MONDO:"):
        cur = str(int(line.strip().split(":")[2]))
    elif line.startswith("xref: MESH:") and cur:
        mesh_to_mondo[line.split()[1][5:]].add(cur)

hgnc = pd.read_csv(HGNC, sep="\t", dtype=str, usecols=["entrez_id", "uniprot_ids"]).dropna()
uniprot_to_entrez = {u: e for e, us in zip(hgnc["entrez_id"], hgnc["uniprot_ids"]) for u in us.split("|")}

GO_TYPE = {"BiologicalProcess": "biological_process", "MolecularActivity": "molecular_function",
           "CellularComponent": "cellular_component"}

def kg_nodes(node):
    """KG node indices of one DrugMechDB node (empty set if PrimeKG has no counterpart)."""
    prefix, local = node["id"].split(":", 1)
    label = node["label"]
    if prefix == "MESH" and label == "Disease":
        return {disease_of_mondo[m] for m in mesh_to_mondo.get(local, ()) if m in disease_of_mondo}
    if prefix == "DB":
        i = by_type["drug"].get(local)
    elif prefix == "UniProt":
        i = by_type["gene/protein"].get(uniprot_to_entrez.get(local))
    elif prefix == "GO" and label in GO_TYPE:
        i = by_type[GO_TYPE[label]].get(str(int(local)))
    elif prefix == "HP":
        i = by_type["effect/phenotype"].get(str(int(local)))
    elif prefix == "UBERON":
        i = by_type["anatomy"].get(str(int(local)))
    elif prefix == "reactome":
        i = by_type["pathway"].get(local)
    else:
        i = None
    return {i} if i is not None else set()

# one entry per (disease, drug) KG pair: curated intermediate nodes, fully mapped paths, curated edges
mech = collections.defaultdict(lambda: {"nodes": set(), "paths": [], "edges": set()})
for p in json.load(open(DMDB)):
    g = p["graph"]
    drug = by_type["drug"].get((g.get("drugbank") or ":").split(":")[1])
    diseases = kg_nodes({"id": g["disease_mesh"], "label": "Disease"})
    if drug is None or not diseases:
        continue
    ends = {g["drug_mesh"], g["drugbank"], g["disease_mesh"]}
    inter = {n["id"]: kg_nodes(n) for n in p["nodes"] if n["id"] not in ends}
    for disease in diseases:
        m = mech[(disease, drug)]
        mapped = set().union(*inter.values()) - {disease, drug} if inter else set()
        m["nodes"] |= mapped
        if inter and all(inter.values()):
            m["paths"].append(mapped)
        to_kg = lambda x: {drug} if x in (g["drug_mesh"], g["drugbank"]) else {disease} if x == g["disease_mesh"] else inter.get(x, set())
        for link in p["links"]:
            m["edges"] |= {frozenset((a, b)) for a in to_kg(link["source"]) for b in to_kg(link["target"]) if a != b}

print(f"{len(mech)} drug-disease pairs, {sum(bool(m['nodes']) for m in mech.values())} with >= 1 mapped mechanism node")

In [ ]:
# ── Subnetwork files: disease node index -> file (file numbers are not node indices) ───
index_csv = os.path.join(OUT_DIR, "subnetwork_file_index.csv")
if os.path.exists(index_csv):
    file_of = pd.read_csv(index_csv).set_index("disease")["file"].to_dict()
else:
    file_of = {}
    for f in glob.glob(os.path.join(SUBNET_DIR, "subnetwork_disease_*.csv")):
        with open(f) as fh:
            rows = csv.reader(fh)
            header = next(rows)
            first = next(rows, None)
        if first:
            file_of[int(first[header.index("pair_id")].split("|")[0])] = f
    pd.DataFrame({"disease": list(file_of), "file": list(file_of.values())}).to_csv(index_csv, index=False)
print(f"{len(file_of)} subnetwork files")

In [ ]:
# ── 1-hop KG neighbourhood of the drugs and diseases (baseline a) ─────────────
ends = {x for pair in mech for x in pair}
neigh = collections.defaultdict(set)
for chunk in pd.read_csv(EDGES, usecols=["x_index", "y_index"], chunksize=2_000_000):
    for a, b in ((chunk["x_index"], chunk["y_index"]), (chunk["y_index"], chunk["x_index"])):
        sel = a.isin(ends)
        for u, v in zip(a[sel], b[sel]):
            neigh[u].add(v)
nodes_of_type = {t: np.array(list(ids.values())) for t, ids in by_type.items()}

In [ ]:
# ── Scores per pair: NetMedGPT subnetwork vs size- and type-matched baselines ──
rng = np.random.default_rng(SEED)

def scores(sub_nodes, sub_edges, m):
    path_hit = [p <= sub_nodes for p in m["paths"]]
    return {"recall": len(m["nodes"] & sub_nodes) / len(m["nodes"]),
            "any_hit": bool(m["nodes"] & sub_nodes),
            "path_recovered": any(path_hit) if path_hit else np.nan,
            "triplet_recall": len(m["edges"] & sub_edges) / len(m["edges"]) if m["edges"] else np.nan}

def sample_like(sub_nodes, pool_of_type):
    """random nodes with the same number of nodes per type as the subnetwork"""
    out = set()
    for t, k in collections.Counter(node_type[n] for n in sub_nodes).items():
        pool = pool_of_type.get(t, np.array([]))
        out |= set(rng.choice(pool, size=min(k, len(pool)), replace=False).tolist()) if len(pool) else set()
    return out

rows = []
by_disease = collections.defaultdict(list)
for (disease, drug), m in mech.items():
    if m["nodes"] and disease in file_of:
        by_disease[disease].append(drug)

for disease, drugs in by_disease.items():
    sub = pd.read_csv(file_of[disease], usecols=["head_index", "tail_index", "pair_id"])
    for drug in drugs:
        g = sub[sub["pair_id"] == f"{disease}|{drug}"]
        if g.empty:
            continue
        m = mech[(disease, drug)]
        sub_nodes = (set(g["head_index"]) | set(g["tail_index"])) - {disease, drug}
        sub_edges = {frozenset(e) for e in zip(g["head_index"], g["tail_index"])}
        row = {"disease": disease, "drug": drug, "n_mech_nodes": len(m["nodes"]), "mech_nodes": "|".join(map(str, sorted(m["nodes"]))), "n_sub_nodes": len(sub_nodes),
               **{f"netmedgpt_{k}": v for k, v in scores(sub_nodes, sub_edges, m).items()}}

        hood = (neigh[drug] | neigh[disease]) - {disease, drug}
        hood_of_type = {t: np.array([n for n in hood if node_type[n] == t]) for t in set(map(node_type.get, sub_nodes))}
        for name, pool in [("neighbourhood", hood_of_type), ("random", nodes_of_type)]:
            s = pd.DataFrame([scores(sample_like(sub_nodes, pool), set(), m) for _ in range(N_RANDOM)])
            row[f"{name}_recall"] = s["recall"].mean()
            row[f"{name}_any_hit"] = s["any_hit"].mean()
            row[f"{name}_path_recovered"] = s["path_recovered"].mean()
        row["neighbourhood_all_recall"] = len(m["nodes"] & hood) / len(m["nodes"])   # whole 1-hop neighbourhood
        row["n_neighbourhood_nodes"] = len(hood)
        rows.append(row)

pairs = pd.DataFrame(rows)
pairs.to_csv(os.path.join(OUT_DIR, "pairs.csv"), index=False)
print(f"{len(pairs)} pairs with a subnetwork and >= 1 mapped mechanism node")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    method: {"mechanism node recall (%)": 100 * pairs[f"{method}_recall"].mean(),
             "pairs with >= 1 mechanism node (%)": 100 * pairs[f"{method}_any_hit"].mean(),
             "fully mapped paths recovered (%)": 100 * pairs[f"{method}_path_recovered"].mean()}
    for method in ["netmedgpt", "neighbourhood", "random"]}).round(2)
summary.loc["curated triplets in subnetwork (%)", "netmedgpt"] = round(100 * pairs["netmedgpt_triplet_recall"].mean(), 2)
for base in ["neighbourhood", "random"]:
    summary.loc["Wilcoxon p (recall, NetMedGPT > baseline)", base] = stats.wilcoxon(
        pairs["netmedgpt_recall"], pairs[f"{base}_recall"], alternative="greater").pvalue
summary.to_csv(os.path.join(OUT_DIR, "summary.csv"))
print(f"pairs: {len(pairs)} | median subnetwork nodes: {pairs['n_sub_nodes'].median():.0f} | "
      f"median 1-hop neighbourhood nodes: {pairs['n_neighbourhood_nodes'].median():.0f} "
      f"(recall of the whole neighbourhood: {100 * pairs['neighbourhood_all_recall'].mean():.1f}%)")
summary